# Sanskrit model — Rāmāyaṇa training (separate run)

**One-click start:** open in Colab, select a GPU under **Runtime → Change runtime type**, then choose **Runtime → Run all** and approve Google Drive access. Use the **same Google account/Drive** as the original training notebook.

This starts from the newest checksummed full checkpoint in `MyDrive/Sanskrit-LLM/colab-1b`, including optimizer state. It trains on five Sanskrit Vālmīki Rāmāyaṇa kāṇḍas from the public-domain-marked [CLTK/Gita Supersite corpus](https://github.com/cltk/sanskrit_text_gitasupersite); this source does **not** include Yuddha or Uttara Kāṇḍa. The text is not independently proofread. It uses verse language modeling, not a guarantee of answering questions or retaining grammar.

The initial target is **10 million additional nonpadding input positions**, repeatedly sampled, not unique tokens. Training uses only these verses and may degrade earlier capabilities. No held-out accuracy or grammar-retention evaluation has been completed, and the notebook does not deploy anything.

New full checkpoints and SHA-256 sidecars are saved every ~5 minutes in **`MyDrive/Sanskrit-LLM/ramayana-v1`**, separate from the original checkpoints. Rerun to resume this run; it will not switch to a newer original checkpoint after it has started. Do not run two copies of this notebook at the same time. Use a separate Colab runtime if keeping the original trainer running: Colab resource limits may disconnect or prevent concurrent runtimes. A disconnect can lose work since the last save; checkpoints accumulate and may fill Drive. GPU availability, quotas and possible costs depend on Colab.


In [ ]:
# One click: Runtime > Run all. Approve Drive access when prompted.
import subprocess
import sys
from pathlib import Path
import torch
from google.colab import drive

TRAINER_COMMIT = "6d0db11b89c6e27e4eec898dde4b1aa007eac293"
CORPUS_COMMIT = "d8a6ccec70099d8f2e23aa3d7fdf439d336b4116"
TARGET_RAMAYANA_POSITIONS = 10_000_000
BATCH_SIZE = 16
SAVE_EVERY_SECONDS = 300
LEARNING_RATE = 1e-5

if not torch.cuda.is_available():
    raise RuntimeError("Choose Runtime > Change runtime type > GPU, then Run all again.")
print("GPU:", torch.cuda.get_device_name(0), flush=True)
drive.mount('/content/drive')

def pinned_checkout(url, revision, directory):
    directory = Path(directory)
    if not directory.exists():
        subprocess.run(['git', 'clone', '--no-checkout', url, str(directory)], check=True)
    if not (directory / '.git').exists():
        raise RuntimeError(f'{directory} exists but is not a Git checkout')
    subprocess.run(['git', '-C', str(directory), 'fetch', '--depth', '1', 'origin', revision], check=True)
    subprocess.run(['git', '-C', str(directory), 'checkout', '--detach', revision], check=True)
    subprocess.run(['git', '-C', str(directory), 'diff', '--exit-code'], check=True)
    subprocess.run(['git', '-C', str(directory), 'diff', '--cached', '--exit-code'], check=True)
    actual = subprocess.check_output(['git', '-C', str(directory), 'rev-parse', 'HEAD'], text=True).strip()
    if actual != revision:
        raise RuntimeError('Wrong checkout revision')
    return directory

repo = pinned_checkout('https://github.com/Yuvii123-bot/HnS.git', TRAINER_COMMIT, '/content/HnS-ramayana')
corpus = pinned_checkout('https://github.com/cltk/sanskrit_text_gitasupersite.git', CORPUS_COMMIT, '/content/ramayana-corpus')
base_dir = Path('/content/drive/MyDrive/Sanskrit-LLM/colab-1b')
out_dir = Path('/content/drive/MyDrive/Sanskrit-LLM/ramayana-v1')
if not base_dir.exists():
    raise RuntimeError(f'Original checkpoint folder missing: {base_dir}. Mount the same Google Drive used for training.')
subprocess.run([sys.executable, str(repo / 'scripts/train_ramayana_colab.py'),
                '--source', str(corpus / 'ramayana'), '--base-dir', str(base_dir),
                '--out-dir', str(out_dir), '--target', str(TARGET_RAMAYANA_POSITIONS),
                '--batch-size', str(BATCH_SIZE), '--save-seconds', str(SAVE_EVERY_SECONDS),
                '--lr', str(LEARNING_RATE)], cwd=str(repo), check=True)
